# LIMEN-RF — Colab reproducibility runner

This notebook verifies that Google Colab can execute the same locked repository state used in WSL and CI.

Before running, create a Colab secret named `GITHUB_TOKEN` containing a GitHub token with **read-only access to this private repository only**. Do not paste the token into a notebook cell.

In [ ]:
REPO = "topassky3/limen-rf"
BRANCH = "chore/infra-0-2"
EXPECTED_COMMIT = "c569554656adb880a2149bfc60ffed66399ede0b"
EXPECTED_DIGEST = "516b25698869a8e0cf8c43549e47d4b6d5f6c1b548421c969540eba43f657ffb"
SEED = 20260914
print("Target commit:", EXPECTED_COMMIT)


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Missing Colab secret GITHUB_TOKEN")

repo_dir = Path("/content/limen-rf")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

auth_home = Path("/tmp/limen-git-home")
auth_home.mkdir(exist_ok=True)
netrc = auth_home / ".netrc"
netrc.write_text(
    f"machine github.com login x-access-token password {token}\n"
)
netrc.chmod(0o600)

env = os.environ.copy()
env["HOME"] = str(auth_home)
env["GIT_TERMINAL_PROMPT"] = "0"

subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        BRANCH,
        "--single-branch",
        f"https://github.com/{REPO}.git",
        str(repo_dir),
    ],
    check=True,
    env=env,
)
subprocess.run(["git", "checkout", EXPECTED_COMMIT], cwd=repo_dir, check=True)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True
).strip()
assert commit == EXPECTED_COMMIT, (commit, EXPECTED_COMMIT)
print("Cloned exact commit:", commit)


In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--locked"], cwd=repo_dir, check=True)
subprocess.run(["uv", "run", "ruff", "check", "."], cwd=repo_dir, check=True)
subprocess.run(["uv", "run", "pytest"], cwd=repo_dir, check=True)


In [ ]:
import json

raw = subprocess.check_output(
    ["uv", "run", "python", "-m", "limen_rf.smoke"],
    cwd=repo_dir,
    text=True,
)
print(raw)
result = json.loads(raw)
assert result["seed"] == SEED
assert result["digest"] == EXPECTED_DIGEST, (result["digest"], EXPECTED_DIGEST)
print("INFRA-0 COLAB CHECK: PASS")
